# ICBHI 2017 Challenge - Audio Preprocessing & Respiratory Cycle Segmentation
## Digital Signal Processing (DSP) Pipeline for Auscultation Analysis

This notebook provides an interactive walkthrough of the audio preprocessing and cycle segmentation pipeline used in the **ICBHI 2017 Respiratory Sound Database**.

### Pipeline Objectives:
1. **Sampling Rate Standardization**: Resample heterogeneous stethoscope recordings (4 kHz, 10 kHz, 44.1 kHz) to a uniform 16 kHz.
2. **Bandpass Filtering**: 4th-order Butterworth bandpass filter (50–2,000 Hz, zero-phase SOS) to eliminate heart sound rumble (<50 Hz) and ambient/transducer noise (>2,000 Hz).
3. **Signal Normalization**: Mean demeaning (DC-offset removal) and peak amplitude normalization to prevent clipping.
4. **Respiratory Cycle Segmentation**: Segment each audio recording into clinical respiratory cycles based on expert start and end annotations.
5. **Quality Verification**: Spectral analysis (Welch PSD), filter frequency response (Bode plot), and cycle inspection.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import soundfile as sf
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 150

### 1. Load Segmented Cycles Metadata
We load the curated metadata table generated by `scripts/preprocess_and_segment.py` containing all 6,898 extracted cycles.

In [ ]:
metadata_path = "../data/processed/segmented_cycles_metadata.csv"
cycles_df = pd.read_csv(metadata_path)
print(f"Total Segmented Cycles: {len(cycles_df):,}")
print("\nClass Distribution:")
display(cycles_df["class"].value_counts().to_frame("Count").assign(Percentage=lambda x: (x["Count"]/len(cycles_df)*100).round(2)))
display(cycles_df.head())

### 2. Digital Butterworth Bandpass Filter Design
Respiratory sounds (vesicular breathing, crackles, wheezes) reside within **50 Hz to 2,000 Hz**.
Heart sounds (S1, S2, murmurs) and body motion rumble occur predominantly **below 50 Hz**.
Electronic noise and ambient room noise dominate **above 2,000 Hz**.
We design a 4th-order Butterworth bandpass filter using Second-Order Sections (SOS) for numerical stability and zero-phase distortion (`sosfiltfilt`).

In [ ]:
fs = 16000
lowcut = 50.0
highcut = 2000.0
order = 4

nyq = 0.5 * fs
sos = signal.butter(order, [lowcut / nyq, highcut / nyq], btype='bandpass', output='sos')

# Compute frequency response
w, h = signal.sosfreqz(sos, worN=2048, fs=fs)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(w, 20 * np.log10(np.maximum(np.abs(h), 1e-5)), color="#2b8a3e", lw=2, label="Filter Magnitude")
ax.axvline(lowcut, color="black", linestyle="--", alpha=0.7, label=f"Low Cutoff: {lowcut:.0f} Hz")
ax.axvline(highcut, color="black", linestyle=":", alpha=0.7, label=f"High Cutoff: {highcut:.0f} Hz")
ax.axhline(-3, color="red", linestyle="--", alpha=0.6, label="-3 dB Cutoff Threshold")
ax.set_title("Butterworth Bandpass Filter Frequency Response (Order 4 SOS)", fontsize=12, fontweight="bold")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude (dB)")
ax.set_xlim(0, 3500)
ax.set_ylim(-60, 5)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### 3. Noise Rejection & Power Spectral Density (PSD)
Comparing the Welch Power Spectral Density before and after filtering on an actual clinical recording.

In [ ]:
# Load an example recording
sample_row = cycles_df[cycles_df["class"] == "Both"].iloc[0]
raw_wav_path = glob.glob(f"../ICBHI_final_database/{sample_row['recording_name']}.wav")[0]
raw_audio, raw_sr = sf.read(raw_wav_path)
if raw_audio.ndim > 1:
    raw_audio = np.mean(raw_audio, axis=1)

# Resample and filter
import soxr
audio_resampled = soxr.resample(raw_audio, raw_sr, fs)
audio_filtered = signal.sosfiltfilt(sos, audio_resampled)
audio_norm = (audio_filtered - np.mean(audio_filtered))
audio_norm = (audio_norm / np.max(np.abs(audio_norm))) * 0.95

# Compute Welch PSD
f_raw, psd_raw = signal.welch(raw_audio, raw_sr, nperseg=min(2048, len(raw_audio)))
f_filt, psd_filt = signal.welch(audio_norm, fs, nperseg=min(2048, len(audio_norm)))

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f_raw[f_raw <= 4000], psd_raw[f_raw <= 4000], color="#4575b4", lw=1.2, alpha=0.7, label=f"Raw Audio ({raw_sr} Hz)")
ax.semilogy(f_filt[f_filt <= 4000], psd_filt[f_filt <= 4000], color="#d73027", lw=1.5, label="Bandpassed Signal (50–2000 Hz)")
ax.axvspan(0, 50, color="gray", alpha=0.2, label="Attenuated: <50 Hz Heart/Motion Noise")
ax.axvspan(2000, 4000, color="gray", alpha=0.15, label="Attenuated: >2000 Hz Transducer Noise")
ax.set_title(f"Welch PSD Noise Rejection: {sample_row['recording_name']}", fontsize=12, fontweight="bold")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power Spectral Density (V²/Hz)")
ax.set_xlim(0, 3000)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### 4. Interactive Cycle Inspection: Waveforms & Spectrograms
Inspect segmented cycles for each acoustic class: **Normal**, **Crackle**, **Wheeze**, and **Both**.

In [ ]:
import librosa
import librosa.display

fig, axes = plt.subplots(4, 2, figsize=(14, 10))
class_colors = {"Normal": "#2b8a3e", "Crackle": "#e67700", "Wheeze": "#d6336c", "Both": "#7048e8"}

for idx, cls in enumerate(["Normal", "Crackle", "Wheeze", "Both"]):
    cyc_row = cycles_df[(cycles_df["class"] == cls) & (cycles_df["cycle_duration"] >= 2.0)].iloc[0]
    cyc_path = os.path.join("../data/processed/segmented_cycles", cyc_row["cycle_file"])
    y_cyc, sr_cyc = sf.read(cyc_path)
    t_cyc = np.linspace(0, len(y_cyc) / sr_cyc, len(y_cyc))
    
    # Plot Waveform
    axes[idx, 0].plot(t_cyc, y_cyc, color=class_colors[cls], lw=0.8)
    axes[idx, 0].set_title(f"{cls} Cycle | Duration: {cyc_row['cycle_duration']:.2f}s | {cyc_row['diagnosis']} ({cyc_row['equipment']})", fontsize=10, fontweight="bold")
    axes[idx, 0].set_ylabel("Amplitude")
    if idx == 3:
        axes[idx, 0].set_xlabel("Time (s)")
        
    # Plot Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y_cyc, sr=sr_cyc, n_fft=1024, hop_length=256, n_mels=64, fmax=2000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(S_dB, sr=sr_cyc, hop_length=256, x_axis='time', y_axis='mel', fmax=2000, ax=axes[idx, 1], cmap='magma')
    axes[idx, 1].set_title(f"Mel-Spectrogram ({cls})", fontsize=10, fontweight="bold")
    if idx == 3:
        axes[idx, 1].set_xlabel("Time (s)")
        
plt.tight_layout()
plt.show()